In [1]:
from ui_generator import generate_ui, DEFAULT_CONFIG
import os
import json
import pathlib
import webbrowser
from string import Template

"""
ui_generator.py

A small universal UI generator: given a configuration dict, it generates a responsive
website/application UI scaffold (index.html, styles.css, script.js) into an output directory.
Includes a preview function that opens the generated HTML in the default browser.

Usage (from a Jupyter cell or script):
generate_ui(DEFAULT_CONFIG, output_dir="site_preview", open_in_browser=True)

This file is self-contained and uses only the Python standard library.
"""


# Default configuration for the universal UI
DEFAULT_CONFIG = {
    "title": "VE Technologies",
    "theme": {
        "primary": "#2563eb",
        "background": "#ffffff",
        "muted": "#6b7280",
        "radius": "12px",
        "font": "Inter, system-ui, -apple-system, 'Segoe UI', Roboto, 'Helvetica Neue', Arial"
    },
    "layout": {
        "show_nav": True,
        "show_hero": True,
        "show_about": True,
        "show_purpose": True,
        "show_services": True,
        "show_form": False,
        "footer_text": "© 2025 VE Technologies Co."
    },
    "brand": {
        "logo_text": "VE",
        "logo_bg": "#2563eb",
        "logo_color": "#fff"
    },
    "content": {
        "hero_title": "VE Technologies",
        "hero_subtitle": "Building innovative solutions for modern business challenges",
        "about_title": "About Us",
        "about_text": "VE Technologies is a forward-thinking company dedicated to delivering cutting-edge technology solutions. We combine innovation with reliability to help businesses thrive in the digital age.",
        "purpose_title": "Our Purpose",
        "purpose_text": "To empower organizations through transformative technology solutions that drive growth, efficiency, and sustainable success in an ever-evolving digital landscape.",
        "services": [
            {
                "title": "Web Development",
                "description": "Custom web applications built with modern frameworks and best practices"
            },
            {
                "title": "Cloud Solutions",
                "description": "Scalable cloud infrastructure and migration services for your business"
            },
            {
                "title": "Digital Consulting",
                "description": "Strategic guidance to help you navigate digital transformation"
            },
            {
                "title": "UI/UX Design",
                "description": "Intuitive user experiences that engage and convert your audience"
            }
        ]
    }
}


HTML_TEMPLATE = Template("""<!doctype html>
<html lang="en">
<head>
  <meta charset="utf-8" />
  <meta name="viewport" content="width=device-width,initial-scale=1" />
  <title>$title</title>
  <link rel="stylesheet" href="styles.css" />
</head>
<body>
  <header class="site-header">
    <div class="brand">
      <div class="logo">$logo_text</div>
      <div class="site-title">$title</div>
    </div>
    $nav_html
  </header>

  $hero_html

  <main class="container">
    $about_html
    $purpose_html
    $services_html
    $form_html
  </main>

  <footer class="site-footer">
    <div>$footer_text</div>
  </footer>

  <script src="script.js"></script>
</body>
</html>
""")

CSS_TEMPLATE = Template(""":root{
  --primary: $primary;
  --bg: $background;
  --muted: $muted;
  --radius: $radius;
  --font: $font;
}
*{box-sizing:border-box}
html,body{height:100%}
body{
  margin:0;
  font-family:var(--font);
  background:var(--bg);
  color:#111827;
  -webkit-font-smoothing:antialiased;
  -moz-osx-font-smoothing:grayscale;
  line-height:1.4;
}
.container{max-width:1100px;margin:2rem auto;padding:0 1rem}
.site-header{
  display:flex;align-items:center;justify-content:space-between;
  padding:1rem;gap:1rem;
  border-bottom:1px solid rgba(15,23,42,0.06);
  background:linear-gradient(90deg, transparent, rgba(0,0,0,0.01));
}
.brand{display:flex;align-items:center;gap:0.75rem}
.logo{
  width:44px;height:44px;border-radius:10px;
  display:flex;align-items:center;justify-content:center;
  color:$logo_color;background:$logo_bg;font-weight:700;
  box-shadow:0 6px 18px rgba(11,116,255,0.08);
}
.site-title{font-weight:700}
.nav{display:flex;gap:0.75rem}
.nav a{color:var(--muted);text-decoration:none;padding:0.5rem 0.75rem;border-radius:6px}
.nav a.primary{background:var(--primary);color:white;padding:0.5rem 0.9rem}

.hero{
  background:linear-gradient(90deg, rgba(11,116,255,0.06), rgba(91,33,182,0.03));
  padding:2.25rem 1rem;
  margin:1rem 0;
  border-radius:var(--radius);
}
.hero .inner{max-width:1000px;margin:0 auto;display:flex;gap:1.25rem;align-items:center;justify-content:space-between}
.hero h1{margin:0;font-size:1.6rem}
.hero p{margin:0;color:var(--muted)}

.cards-grid{
  display:grid;
  grid-template-columns:repeat(auto-fit,minmax(220px,1fr));
  gap:1rem;
  margin-bottom:1.5rem;
}
.card{
  background:white;padding:1rem;border-radius:calc(var(--radius));
  box-shadow:0 6px 18px rgba(16,24,40,0.04);
}
.card h3{margin:0 0 0.5rem 0}
.card p{margin:0;color:var(--muted);font-size:0.95rem}

.form-card{
  background:white;padding:1rem;border-radius:calc(var(--radius));
  box-shadow:0 6px 18px rgba(16,24,40,0.04);max-width:640px;margin:0 auto;
}
.form-row{display:flex;gap:0.5rem;margin-bottom:0.5rem}
.form-row input, .form-row textarea{
  flex:1;padding:0.6rem;border:1px solid rgba(15,23,42,0.06);border-radius:6px;font-size:0.95rem
}
.btn{
  display:inline-block;background:var(--primary);color:white;padding:0.6rem 0.9rem;border-radius:8px;border:none;cursor:pointer
}
.content-section{
  margin:3rem 0;
  max-width:800px;
  margin-left:auto;
  margin-right:auto;
  text-align:center;
}
.content-section h2{
  font-size:2rem;
  margin-bottom:1rem;
  color:#111827;
}
.section-text{
  font-size:1.125rem;
  line-height:1.6;
  color:var(--muted);
}
.services-grid{
  display:grid;
  grid-template-columns:repeat(auto-fit,minmax(280px,1fr));
  gap:1.5rem;
  margin-top:2rem;
}
.service-card{
  background:white;
  padding:2rem;
  border-radius:var(--radius);
  box-shadow:0 4px 12px rgba(16,24,40,0.08);
  text-align:left;
  transition:transform 0.2s ease, box-shadow 0.2s ease;
}
.service-card:hover{
  transform:translateY(-2px);
  box-shadow:0 8px 24px rgba(16,24,40,0.12);
}
.service-card h3{
  margin:0 0 1rem 0;
  color:#111827;
  font-size:1.25rem;
}
.service-card p{
  margin:0;
  color:var(--muted);
  line-height:1.5;
}
.site-footer{padding:1rem;text-align:center;color:var(--muted);font-size:0.95rem;margin-top:2rem}
@media (max-width:700px){
  .hero .inner{flex-direction:column;align-items:flex-start}
  .content-section h2{font-size:1.75rem}
  .services-grid{grid-template-columns:1fr}
}
""")

JS_TEMPLATE = """// Minimal interactivity for the generated UI
document.addEventListener('click', (e) => {
  if (e.target.matches('[data-action="cta"]')) {
    const src = e.target.closest('[data-src]');
    if (src) {
      // Example: toggle active state
      src.classList.toggle('active');
      e.target.innerText = src.classList.contains('active') ? 'Open' : 'Open';
    } else {
      alert('CTA clicked!');
    }
  }
});
"""

def _render_nav(show_nav: bool) -> str:
    if not show_nav:
        return ""
    return '<nav class="nav"><a href="#about">About</a><a href="#purpose">Purpose</a><a href="#services">Services</a><a href="#contact" class="primary">Contact</a></nav>'

def _render_hero(show_hero: bool, hero_title: str, hero_subtitle: str) -> str:
    if not show_hero:
        return ""
    return f'''
    <section class="hero">
      <div class="inner">
        <div>
          <h1>{hero_title}</h1>
          <p>{hero_subtitle}</p>
        </div>
        <div>
          <button class="btn" data-action="cta">Get Started</button>
        </div>
      </div>
    </section>
    '''

def _render_cards(count: int) -> str:
    cards = []
    for i in range(1, max(1, count) + 1):
        cards.append(f'''
        <article class="card" data-src="card-{i}">
          <h3>Component {i}</h3>
          <p>Reusable UI component pattern for lists, profiles, or feature highlights.</p>
        </article>
        ''')
    return "\n".join(cards)

def _render_form(show_form: bool) -> str:
    if not show_form:
        return ""
    return '''
    <section class="form-card" aria-label="Contact form">
      <h3>Try it out</h3>
      <p style="color:var(--muted);margin-bottom:0.75rem">Send a quick message to test styles and spacing.</p>
      <div class="form-row">
        <input type="text" placeholder="Name" />
        <input type="email" placeholder="Email" />
      </div>
      <div class="form-row">
        <textarea rows="4" placeholder="Message"></textarea>
      </div>
      <div style="text-align:right"><button class="btn">Send</button></div>
    </section>
    '''

def _render_about(show_about: bool, title: str, text: str) -> str:
    if not show_about:
        return ""
    return f'''
    <section id="about" class="content-section">
      <h2>{title}</h2>
      <p class="section-text">{text}</p>
    </section>
    '''

def _render_purpose(show_purpose: bool, title: str, text: str) -> str:
    if not show_purpose:
        return ""
    return f'''
    <section id="purpose" class="content-section">
      <h2>{title}</h2>
      <p class="section-text">{text}</p>
    </section>
    '''

def _render_services(show_services: bool, services: list) -> str:
    if not show_services or not services:
        return ""
    
    service_cards = []
    for service in services:
        service_cards.append(f'''
        <div class="service-card">
          <h3>{service.get("title", "")}</h3>
          <p>{service.get("description", "")}</p>
        </div>
        ''')
    
    return f'''
    <section id="services" class="content-section">
      <h2>Our Services</h2>
      <div class="services-grid">
        {"".join(service_cards)}
      </div>
    </section>
    '''

def generate_ui(config: dict = None, output_dir: str = "ui_output", open_in_browser: bool = False) -> str:
    """
    Generate the UI scaffold files in output_dir.
    Returns the path to the generated index.html.
    """
    cfg = dict(DEFAULT_CONFIG)
    if config:
        # deep-ish merge for top-level keys
        for k, v in config.items():
            if isinstance(v, dict) and k in cfg:
                cfg[k].update(v)
            else:
                cfg[k] = v

    out = pathlib.Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)

    nav_html = _render_nav(cfg.get("layout", {}).get("show_nav", True))
    hero_html = _render_hero(
        cfg.get("layout", {}).get("show_hero", True), 
        cfg.get("content", {}).get("hero_title", cfg.get("title", "Universal UI")),
        cfg.get("content", {}).get("hero_subtitle", "Building innovative solutions")
    )
    about_html = _render_about(
        cfg.get("layout", {}).get("show_about", True),
        cfg.get("content", {}).get("about_title", "About Us"),
        cfg.get("content", {}).get("about_text", "")
    )
    purpose_html = _render_purpose(
        cfg.get("layout", {}).get("show_purpose", True),
        cfg.get("content", {}).get("purpose_title", "Our Purpose"),
        cfg.get("content", {}).get("purpose_text", "")
    )
    services_html = _render_services(
        cfg.get("layout", {}).get("show_services", True),
        cfg.get("content", {}).get("services", [])
    )
    form_html = _render_form(cfg.get("layout", {}).get("show_form", True))

    html = HTML_TEMPLATE.substitute(
        title=cfg.get("title", "Universal UI"),
        logo_text=cfg.get("brand", {}).get("logo_text", "UI"),
        nav_html=nav_html,
        hero_html=hero_html,
        about_html=about_html,
        purpose_html=purpose_html,
        services_html=services_html,
        form_html=form_html,
        footer_text=cfg.get("layout", {}).get("footer_text", "")
    )

    css = CSS_TEMPLATE.substitute(
        primary=cfg["theme"]["primary"],
        background=cfg["theme"]["background"],
        muted=cfg["theme"]["muted"],
        radius=cfg["theme"]["radius"],
        font=cfg["theme"]["font"],
        logo_color=cfg["brand"]["logo_color"],
        logo_bg=cfg["brand"]["logo_bg"],
    )

    js = JS_TEMPLATE

    (out / "index.html").write_text(html, encoding="utf-8")
    (out / "styles.css").write_text(css, encoding="utf-8")
    (out / "script.js").write_text(js, encoding="utf-8")
    # Write the config used for reproducibility
    (out / "ui_config.json").write_text(json.dumps(cfg, indent=2), encoding="utf-8")

    index_path = str((out / "index.html").resolve())
    if open_in_browser:
        webbrowser.open(f"file://{index_path}")
    return index_path


if __name__ == "__main__":
    # When executed directly, generate with default config and open preview.
    print("Generating UI scaffold (default config)...")
    path = generate_ui(DEFAULT_CONFIG, output_dir="site_preview", open_in_browser=True)
    print(f"Generated static preview at: {path}")

ModuleNotFoundError: No module named 'ui_generator'